
# PCA vs t-SNE (2D y 3D) — Heart Disease UCI (Kaggle)

Este cuaderno implementa un **ejercicio completo** para comparar **PCA** y **t-SNE** (2D y 3D) sobre el dataset **Heart Disease UCI** de Kaggle.

> **Requisitos previos**  
> - Archivo `heart.csv` (descárgalo desde Kaggle y colócalo junto a este notebook).  
> - Paquetes: `pandas`, `numpy`, `matplotlib`, `scikit-learn`  
>   ```bash
>   pip install pandas numpy matplotlib scikit-learn
>   ```

**Objetivos:**
1. Cargar y escalar los datos clínicos.
2. Aplicar **PCA** (2D y 3D) y **t-SNE** (2D y 3D).
3. Calcular métricas: **varianza explicada** (PCA), **trustworthiness** y **KNN (5-fold)**.
4. Visualizar resultados y comparar tiempos.


In [ ]:

# ====== Imports ======
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Activar proyección 3D en Matplotlib
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier

# Configuración general de gráficos
plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = False



## 1) Cargar datos

- Descarga el dataset desde Kaggle: **Heart Disease UCI**.  
- Asegúrate de tener el archivo **`heart.csv`** en la misma carpeta que este notebook.


In [ ]:

# ====== Cargar dataset ======
csv_path = "heart.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "No se encontró 'heart.csv'. Descárgalo desde Kaggle (Heart Disease UCI) y colócalo junto a este notebook."
    )

df = pd.read_csv(csv_path)
display(df.head())
print("Shape:", df.shape)
print("Columnas:", list(df.columns))



### Selección de características sugeridas
Para el reto, selecciona al menos **5 características** relevantes (puedes ajustar según tu criterio clínico):

- `age` (edad)  
- `trestbps` (presión arterial en reposo)  
- `chol` (colesterol sérico)  
- `thalach` (frecuencia cardiaca máxima)  
- `oldpeak` (depresión del ST inducida por ejercicio)  

> **Nota**: La etiqueta objetivo es `target` (0 = sin enfermedad, 1 = con enfermedad).


In [ ]:

# ====== Separar X (features) e y (target) y escalar ======
target_col = "target"
if target_col not in df.columns:
    raise KeyError(f"No se encontró la columna '{target_col}' en el CSV. Revisa el archivo.")

# Selección de columnas (puedes añadir/eliminar columnas numéricas según necesites)
feature_cols = [c for c in df.columns if c != target_col]

X = df[feature_cols].values
y = df[target_col].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("X_scaled shape:", X_scaled.shape, "| y shape:", y.shape)



## 2) PCA (2D y 3D)

- Reducimos a 2 y 3 componentes.  
- Obtenemos **varianza explicada acumulada**.  
- Generamos **dispersión 2D/3D** coloreada por `y`.


In [ ]:

# ====== PCA 2D ======
t0 = time.perf_counter()
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)
pca2_time = time.perf_counter() - t0
pca2_var = pca2.explained_variance_ratio_.sum()

print(f"Tiempo PCA 2D: {pca2_time:.4f}s | Varianza acumulada 2D: {pca2_var:.4f}")

plt.figure()
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y, s=14, alpha=0.85)
plt.title(f"PCA 2D — Varianza: {pca2_var:.3f}")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout()
plt.show()


In [ ]:

# ====== PCA 3D ======
t0 = time.perf_counter()
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(X_scaled)
pca3_time = time.perf_counter() - t0
pca3_var = pca3.explained_variance_ratio_.sum()

print(f"Tiempo PCA 3D: {pca3_time:.4f}s | Varianza acumulada 3D: {pca3_var:.4f}")

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2], c=y, s=14, alpha=0.85)
ax.set_title(f"PCA 3D — Varianza: {pca3_var:.3f}")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
plt.tight_layout()
plt.show()



## 3) t-SNE (2D y 3D)

Parámetros sugeridos (compatibles con versiones antiguas de `scikit-learn`):
- `perplexity = 20`
- `learning_rate = 200`
- `init = "pca"`
- `early_exaggeration = 8`


In [ ]:

# ====== t-SNE 2D ======
tsne_params = dict(
    perplexity=20,
    learning_rate=200,
    init="pca",
    early_exaggeration=8,
    random_state=42,
)

t0 = time.perf_counter()
tsne2 = TSNE(n_components=2, **tsne_params)
X_tsne2 = tsne2.fit_transform(X_scaled)
tsne2_time = time.perf_counter() - t0

print(f"Tiempo t-SNE 2D: {tsne2_time:.4f}s")

plt.figure()
plt.scatter(X_tsne2[:, 0], X_tsne2[:, 1], c=y, s=14, alpha=0.85)
plt.title("t-SNE 2D")
plt.xlabel("Dim 1"); plt.ylabel("Dim 2")
plt.tight_layout()
plt.show()


In [ ]:

# ====== t-SNE 3D ======
t0 = time.perf_counter()
tsne3 = TSNE(n_components=3, **tsne_params)
X_tsne3 = tsne3.fit_transform(X_scaled)
tsne3_time = time.perf_counter() - t0

print(f"Tiempo t-SNE 3D: {tsne3_time:.4f}s")

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_tsne3[:, 0], X_tsne3[:, 1], X_tsne3[:, 2], c=y, s=14, alpha=0.85)
ax.set_title("t-SNE 3D")
ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2"); ax.set_zlabel("Dim 3")
plt.tight_layout()
plt.show()



## 4) Métricas de comparación

- **Varianza acumulada** (solo PCA).  
- **Trustworthiness** (PCA 2D/3D y t-SNE 2D/3D; k=5).  
- **Exactitud KNN (5-fold)** en cada espacio: original (escalado), PCA 2D/3D, t-SNE 2D/3D.


In [ ]:

# ====== Trustworthiness ======
tw_pca2  = trustworthiness(X_scaled, X_pca2, n_neighbors=5)
tw_pca3  = trustworthiness(X_scaled, X_pca3, n_neighbors=5)
tw_tsne2 = trustworthiness(X_scaled, X_tsne2, n_neighbors=5)
tw_tsne3 = trustworthiness(X_scaled, X_tsne3, n_neighbors=5)

print(f"Trustworthiness — PCA2D: {tw_pca2:.3f} | PCA3D: {tw_pca3:.3f} | tSNE2D: {tw_tsne2:.3f} | tSNE3D: {tw_tsne3:.3f}")


In [ ]:

# ====== KNN (5-fold) ======
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

acc_orig  = cross_val_score(knn, X_scaled, y, cv=cv, scoring="accuracy")
acc_pca2  = cross_val_score(knn, X_pca2,  y, cv=cv, scoring="accuracy")
acc_pca3  = cross_val_score(knn, X_pca3,  y, cv=cv, scoring="accuracy")
acc_tsne2 = cross_val_score(knn, X_tsne2, y, cv=cv, scoring="accuracy")
acc_tsne3 = cross_val_score(knn, X_tsne3, y, cv=cv, scoring="accuracy")

summary = pd.DataFrame({
    "Método": [
        f"Original ({X_scaled.shape[1]}D, n={X_scaled.shape[0]})",
        "PCA 2D", "PCA 3D", "t-SNE 2D", "t-SNE 3D"
    ],
    "Dimensiones": [X_scaled.shape[1], 2, 3, 2, 3],
    "Tiempo_fit (s)": [np.nan, pca2_time, pca3_time, tsne2_time, tsne3_time],
    "Varianza acumulada (PCA)": [np.nan, pca2_var, pca3_var, np.nan, np.nan],
    "Trustworthiness (k=5)": [np.nan, tw_pca2, tw_pca3, tw_tsne2, tw_tsne3],
    "KNN Acc media (5-fold)": [
        acc_orig.mean(), acc_pca2.mean(), acc_pca3.mean(), acc_tsne2.mean(), acc_tsne3.mean()
    ],
    "KNN Acc std (5-fold)": [
        acc_orig.std(), acc_pca2.std(), acc_pca3.std(), acc_tsne2.std(), acc_tsne3.std()
    ],
})

summary


## 5) Reflexión

- ¿Qué diferencias observas entre PCA (2D/3D) y t-SNE (2D/3D) en este dataset?
- ¿Cómo cambian las métricas (trustworthiness, KNN) entre espacios?
- ¿Qué parámetros de t-SNE (por ejemplo, `perplexity`) probarías para mejorar la separación visual?
- ¿Usarías PCA con más componentes (p.ej., 10) para un modelo supervisado? ¿Por qué?
